# 02 - Split raw data into train / test (80/20)
**Mục tiêu:** chia `data_raw/auto-mpg.csv` (dữ liệu **thô**, chưa tiền xử lý — đã khảo sát trong `01_data_overview.ipynb`) thành 2 tập `train_raw` và `test_raw` theo tỉ lệ **80/20**, thực hiện ngay **trước bất kỳ bước EDA hay tiền xử lý nào**.

**Ràng buộc buổi đầu thầy yêu cầu:**
- **Không sử dụng thư viện scikit-learn** cho phần cài đặt thuật toán chính.
- **Tập test thực tế luôn là tập thô** — không được tiền xử lý hay tham gia vào quá trình fit thông số.
- **Sử dụng các kỹ thuật chống rò rỉ dữ liệu (data leakage prevention)** trong toàn bộ pipeline.

**Giải pháp của nhóm đề ra:**
| Ràng buộc | Giải pháp |
| :--- | :--- |
| Không dùng scikit-learn | Chỉ sử dụng `scikit-learn` để **so sánh kết quả**. Toàn bộ thuật toán, pipeline tiền xử lý được cài đặt thuần NumPy/Pandas. |
| Tập test thực tế là tập thô | **Split tập thô đầu tiên** (Notebook 01/02) để chia 2 tập train/test ở dạng thô. Tập test được giữ độc lập hoàn toàn. Nếu cần tập val ở evaluation để chọn hyperparameters thì split val từ train. |
| Chống rò rỉ dữ liệu | **Chỉ fit tham số duy nhất trên tập train**, xuất ra file `.pkl` để có thể tái sử dụng biến đổi dữ liệu mới trong tương lai một cách nhất quán. Val và test tuyệt đối không tham gia. |

## Vì sao cần split trước EDA và Feature Engineering?
- **Ngăn chặn Data Leakage (Rò rỉ dữ liệu):** Nếu tính toán thống kê (Mean/Std, Median, IQR bounds, Min/Max) trên toàn bộ dataset trước khi chia, tập test sẽ bị "nhìn trộm". Điều đó khiến việc đánh giá mô hình trên test không còn phản ánh đúng khả năng tổng quát hóa thực tế.
- **Tập test hoàn toàn độc lập:** Tập test được lưu ở dạng thô (`test_raw.csv`), tuyệt đối không tham gia vào quá trình fit hay chọn siêu tham số.
- **Tách tập Validation khi cần:** Trong quá trình đánh giá và chọn bậc đa thức (hyperparameter tuning), tập validation sẽ được tách ra từ chính tập `train`, đảm bảo tính toàn vẹn của tập test.

**Nội dung thực hiện:**
1. Đọc dữ liệu thô, giữ nguyên toàn bộ giá trị ban đầu
2. Loại bỏ các dòng hoàn toàn rỗng
3. Loại bỏ các dòng trùng lặp hoàn toàn
4. Hàm phân chia Train/Test thuần Pandas + NumPy (random, fixed seed 42)
5. Kiểm tra kích thước và không trùng lặp index (overlap = 0)
6. Kiểm tra tính đại diện phân phối giữa Train và Test
7. Kiểm tra tính tiền định (determinism / reproducibility)
8. Lưu kết quả ra `data_processed/train_raw.csv` và `test_raw.csv`

In [1]:
# Import libraries - use ONLY plain pandas/numpy, NOT sklearn
from typing import List, Tuple
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RAW_PATH = '../data_raw/auto-mpg.csv'
OUT_DIR = '../data_processed'
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.80
TEST_RATIO = 0.20
assert abs((TRAIN_RATIO + TEST_RATIO) - 1.0) < 1e-9, 'Tỉ lệ train + test phải bằng 1.0'

## 1. Đọc dữ liệu thô
Đọc với `dtype=str, keep_default_na=False, na_values=[]` để pandas **không tự động biến đổi** dữ liệu. Toàn bộ chuỗi thô được giữ nguyên vẹn.

In [2]:
df_raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False, na_values=[])
df_raw = df_raw.reset_index(drop=False).rename(columns={'index': 'orig_row_id'})

n_raw = df_raw.shape[0]
print(f'Số dòng đọc từ file gốc: {n_raw}')
df_raw.head()

Số dòng đọc từ file gốc: 407


,orig_row_id,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,0,33.5,4,85.0,70,-1945,16.8,77.0,3,Datsun F-10 Hatchback
1,1,35.7,4,98.0,80,1915,14.4,79.0,USA,Dodge Colt Hatchback Custom
2,2,36.0,4,135.0,84,2370,13.0,82.0,1,dodge charger 2.2
3,3,32.9,4,119.0,100,2615,14.8,81.0,3,DATSUN 200SX
4,4,32.0,4,91.0,67,1965,15.7,82.0,3,honda civic (auto)


## 2. Loại bỏ dòng hoàn toàn rỗng
Dòng rỗng toàn bộ các cột không mang bất kỳ thông tin nào, xuất hiện do lỗi xuất file.

In [3]:
def is_row_completely_empty(df: pd.DataFrame, cols: List[str]) -> pd.Series:
    empty_variants = {'', 'nan', 'NaN', 'NAN', 'None', 'none', 'null', 'NULL'}
    tmp = df[cols].astype(str).apply(lambda s: s.str.strip())
    mask = tmp.apply(lambda s: s.isin(empty_variants)).all(axis=1)
    return mask

data_cols = [c for c in df_raw.columns if c != 'orig_row_id']
mask_empty = is_row_completely_empty(df_raw, data_cols)

print(f'Số dòng rỗng hoàn toàn bị loại: {mask_empty.sum()}')
df_clean_1 = df_raw.loc[~mask_empty].copy()
print(f'Số dòng còn lại: {df_clean_1.shape[0]} (trước đó: {n_raw})')

Số dòng rỗng hoàn toàn bị loại: 3
Số dòng còn lại: 404 (trước đó: 407)


## 3. Loại bỏ dòng trùng lặp hoàn toàn
Nếu không loại trùng lặp trước khi split, một mẫu có thể vừa rơi vào train vừa rơi vào test, dẫn đến đánh giá sai lệch.

In [4]:
mask_dup = df_clean_1[data_cols].duplicated(keep='first')
print(f'Số dòng trùng lặp hoàn toàn bị loại: {mask_dup.sum()}')

df_clean = df_clean_1.loc[~mask_dup].copy()
df_clean = df_clean.reset_index(drop=True)

print(f'Số dòng còn lại sau khi lọc: {df_clean.shape[0]}')
print(f'Tổng số dòng đã loại (rỗng + trùng): {n_raw - df_clean.shape[0]} / {n_raw}')

Số dòng trùng lặp hoàn toàn bị loại: 1
Số dòng còn lại sau khi lọc: 403
Tổng số dòng đã loại (rỗng + trùng): 4 / 407


## 4. Hàm phân chia Train / Test (80/20)
- Trộn ngẫu nhiên index bằng `np.random.seed(SEED)` đảm bảo tái lập 100%.
- Cắt theo tỉ lệ 80% Train và 20% Test.
- Trả về 2 DataFrame độc lập, **không trùng lặp index**.

In [5]:
def split_train_test(
    df: pd.DataFrame,
    train_ratio: float = 0.8,
    seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    assert 0 < train_ratio < 1, 'train_ratio phải nằm trong (0, 1)'

    n = len(df)
    np.random.seed(seed)
    shuffled_idx = np.random.permutation(n)

    n_train = int(np.floor(n * train_ratio))
    n_test = n - n_train

    idx_train = shuffled_idx[:n_train]
    idx_test = shuffled_idx[n_train:]

    train_df = df.iloc[idx_train].reset_index(drop=True).copy()
    test_df = df.iloc[idx_test].reset_index(drop=True).copy()

    return train_df, test_df

train_raw, test_raw = split_train_test(
    df_clean, train_ratio=TRAIN_RATIO, seed=SEED
)

print('Phân chia Train / Test (80/20) hoàn tất.')

Phân chia Train / Test (80/20) hoàn tất.


## 5. Kiểm tra kích thước và tính độc lập (Không Overlap)

In [6]:
print(f'Tổng số dòng đưa vào split: {df_clean.shape[0]}')
print(f'train_raw: {train_raw.shape}')
print(f'test_raw : {test_raw.shape}')
print(f'Tổng 2 tập: {train_raw.shape[0] + test_raw.shape[0]}')

actual_ratio = {
    'train': round(train_raw.shape[0] / df_clean.shape[0], 3),
    'test': round(test_raw.shape[0] / df_clean.shape[0], 3),
}
print(f'Tỉ lệ thực tế: {actual_ratio}')

Tổng số dòng đưa vào split: 403
train_raw: (322, 10)
test_raw : (81, 10)
Tổng 2 tập: 403
Tỉ lệ thực tế: {'train': 0.799, 'test': 0.201}


In [7]:
set_train = set(train_raw['orig_row_id'])
set_test = set(test_raw['orig_row_id'])

overlap = set_train & set_test
print(f'Số dòng trùng nhau giữa train và test: {len(overlap)}')
assert len(overlap) == 0, 'LỖI: train và test có dòng bị trùng lặp!'
assert (set_train | set_test) == set(df_clean['orig_row_id']), 'LỖI: Hợp của train và test không bằng df_clean!'

print('=> Tuyệt đối KHÔNG có overlap giữa Train và Test. Bảo toàn đầy đủ dữ liệu.')

Số dòng trùng nhau giữa train và test: 0
=> Tuyệt đối KHÔNG có overlap giữa Train và Test. Bảo toàn đầy đủ dữ liệu.


## 6. Kiểm tra tính đại diện phân phối giữa Train và Test

In [8]:
def quick_stats(series_str: pd.Series) -> pd.Series:
    num = pd.to_numeric(series_str, errors='coerce')
    return num.describe()[['count', 'mean', 'std', 'min', '50%', 'max']]

mpg_comparison = pd.DataFrame({
    'train': quick_stats(train_raw['mpg']),
    'test': quick_stats(test_raw['mpg']),
}).round(2)

print('Thống kê mô tả mpg trên 2 tập:')
display(mpg_comparison)

Thống kê mô tả mpg trên 2 tập:


,train,test
count,317.00,78.00
mean,23.38,23.74
std,7.79,8.09
min,10.00,9.00
50%,22.30,23.50
max,44.60,46.60


In [9]:
def value_ratio(series_str: pd.Series) -> pd.Series:
    return series_str.astype(str).str.strip().value_counts(normalize=True).round(3)

origin_comparison = pd.DataFrame({
    'train': value_ratio(train_raw['origin']),
    'test': value_ratio(test_raw['origin']),
}).fillna(0.0)

print("Tỉ lệ phân bố 'origin' trên 2 tập:")
display(origin_comparison)

Tỉ lệ phân bố 'origin' trên 2 tập:


,train,test
origin,,
1,0.606,0.593
2,0.168,0.185
3,0.205,0.160
Europe,0.006,0.012
USA,0.016,0.049


In [10]:
cylinders_comparison = pd.DataFrame({
    'train': value_ratio(train_raw['cylinders']),
    'test': value_ratio(test_raw['cylinders']),
}).fillna(0.0)

print("Tỉ lệ phân bố 'cylinders' trên 2 tập:")
display(cylinders_comparison)

Tỉ lệ phân bố 'cylinders' trên 2 tập:


,train,test
cylinders,,
3,0.009,0.012
4,0.503,0.494
5,0.009,0.012
6,0.196,0.247
8,0.267,0.235
four,0.009,0.000
six,0.006,0.000


## 7. Kiểm tra tính tiền định (Determinism / Reproducibility)

In [11]:
train_raw_2, test_raw_2 = split_train_test(
    df_clean, train_ratio=TRAIN_RATIO, seed=SEED
)

assert train_raw.equals(train_raw_2) and test_raw.equals(test_raw_2), 'Lỗi: Split không cố định được kết quả!'
print('=> Hàm split tiền định 100% khi dùng cùng seed.')

=> Hàm split tiền định 100% khi dùng cùng seed.


## 8. Lưu kết quả ra `data_processed/`

In [12]:
cols_to_save = data_cols

train_out_path = os.path.join(OUT_DIR, 'train_raw.csv')
test_out_path = os.path.join(OUT_DIR, 'test_raw.csv')

train_raw[cols_to_save].to_csv(train_out_path, index=False)
test_raw[cols_to_save].to_csv(test_out_path, index=False)

print('Đã lưu:')
print(f'  {train_out_path} -> shape {train_raw[cols_to_save].shape}')
print(f'  {test_out_path}  -> shape {test_raw[cols_to_save].shape}')

Đã lưu:
  ../data_processed\train_raw.csv -> shape (322, 9)
  ../data_processed\test_raw.csv  -> shape (81, 9)


In [13]:
# Kiểm tra lại file đã lưu
reloaded_train = pd.read_csv(train_out_path, dtype=str, keep_default_na=False, na_values=[])
reloaded_test = pd.read_csv(test_out_path, dtype=str, keep_default_na=False, na_values=[])

assert 'orig_row_id' not in reloaded_train.columns and 'orig_row_id' not in reloaded_test.columns
assert list(reloaded_train.columns) == cols_to_save
print(f'Re-read train: {reloaded_train.shape}, test: {reloaded_test.shape}')
print('=> Cột và định dạng được bảo toàn nguyên vẹn.')

Re-read train: (322, 9), test: (81, 9)
=> Cột và định dạng được bảo toàn nguyên vẹn.


## Kết luận
- Đã phân chia dữ liệu thô ban đầu thành 2 tập `train_raw` (80% ~ 322 dòng) và `test_raw` (20% ~ 81 dòng).
- Tập test được cô lập hoàn toàn ở trạng thái thô để chống data leakage.
- Không tách riêng tập validation ở bước dữ liệu thô này; nếu cần dùng validation để đánh giá/chọn siêu tham số (bậc đa thức), tập val sẽ được tách trực tiếp từ `train` ở bước evaluation.